In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# === BOOTSTRAP: RUN THIS FIRST ===
import os, sys
from pathlib import Path
REPO_PATH = Path("/content/drive/MyDrive/MaintainAI/code")
os.chdir(REPO_PATH)
sys.path.insert(0, str(REPO_PATH))
print("ROOT:", REPO_PATH)
print("Exists:", REPO_PATH.exists())
print("src exists:", (REPO_PATH / "src").exists())

ROOT: /content/drive/MyDrive/MaintainAI/code
Exists: True
src exists: True


# 05 — SLM dataset creation (grounded, rule-derived)
No hand-written answers. Targets derive from measured pipeline quantities.

Each example contains:
- `system`: system prompt
- `user`: rendered machine context (via ContextBuilder — same as inference)
- `target`: structured JSON validated by `SLMAnalysis` schema
- `meta`: engine, cycle, true RUL, source, model_version

Engine separation:
- SLM-train ← predictive train engines (80)
- SLM-val ← predictive val engines (20)
- SLM-test ← NASA test engines (100), namespaced as `T###`

In [3]:
from src.slm_dataset import generate_dataset

# Generate full dataset (this may take a minute)
summary = generate_dataset(
    raw_dir='CMAPSSData',
    artifact_dir='models/predictive',
    out_dir='data/slm',
    per_engine_train=8,
    per_engine_val=7,
    per_engine_test=1,
    seed=42
)
print('Dataset summary:', summary)
print('Train:', summary['train'], '| Val:', summary['val'], '| Test:', summary['test'])

Dataset summary: {'train': 640, 'val': 140, 'test': 100}
Train: 640 | Val: 140 | Test: 100


In [4]:
# Inspect a few examples
import json

for split in ['train', 'val', 'test']:
    with open(f'data/slm/{split}.jsonl') as f:
        first = json.loads(f.readline())
    print(f'\n=== {split.upper()} (first example) ===')
    print('System:', first['system'][:100], '...')
    print('User (first 200 chars):', first['user'][:200], '...')
    print('Target:', first['target'])
    print('Meta:', first['meta'])
    print()


=== TRAIN (first example) ===
System: You are a predictive maintenance assistant. Analyze machine condition ONLY from the provided evidenc ...
User (first 200 chars): You are a predictive maintenance assistant.
Analyze machine condition ONLY from the provided evidence.
Do not invent sensor measurements, maintenance history, inspections,
or unsupported failure modes ...
Target: {"risk_level": "LOW", "likely_condition": "healthy_operation", "confidence": 0.85, "evidence": ["rul_healthy_margin", "failure_probability_low"], "recommended_action": "continue_monitoring", "urgency": "ROUTINE"}
Meta: {'engine': 'M031', 'cycle': 12, 'true_rul': 222.0, 'source': 'derived-train-engines', 'model_version': 'rul-xgb-clf-v1'}


=== VAL (first example) ===
System: You are a predictive maintenance assistant. Analyze machine condition ONLY from the provided evidenc ...
User (first 200 chars): You are a predictive maintenance assistant.
Analyze machine condition ONLY from the provided evidence.
Do not in

In [5]:
# Verify schema validation on all examples
from src.schemas import SLMAnalysis

for split in ['train', 'val', 'test']:
    count = 0
    with open(f'data/slm/{split}.jsonl') as f:
        for line in f:
            ex = json.loads(line)
            SLMAnalysis.model_validate_json(ex['target'])
            count += 1
    print(f'{split}: {count} examples — all schema valid ✓')

train: 640 examples — all schema valid ✓
val: 140 examples — all schema valid ✓
test: 100 examples — all schema valid ✓


In [6]:
# Distribution analysis
from collections import Counter

for split in ['train', 'val', 'test']:
    risk_cnt = Counter()
    cond_cnt = Counter()
    with open(f'data/slm/{split}.jsonl') as f:
        for line in f:
            ex = json.loads(line)
            target = json.loads(ex['target'])
            risk_cnt[target['risk_level']] += 1
            cond_cnt[target['likely_condition']] += 1
    print(f'{split}: risk={dict(risk_cnt)} | condition={dict(cond_cnt)}')

train: risk={'LOW': 480, 'CRITICAL': 154, 'HIGH': 6} | condition={'healthy_operation': 375, 'unknown_anomaly': 140, 'hpc_degradation': 125}
val: risk={'LOW': 100, 'CRITICAL': 34, 'MEDIUM': 3, 'HIGH': 3} | condition={'healthy_operation': 77, 'unknown_anomaly': 31, 'hpc_degradation': 32}
test: risk={'LOW': 75, 'MEDIUM': 2, 'CRITICAL': 20, 'HIGH': 3} | condition={'healthy_operation': 52, 'hpc_degradation': 27, 'unknown_anomaly': 21}


In [7]:
# Verify engine separation (no leakage)
train_engines = set()
val_engines = set()
test_engines = set()

for split, engines in [('train', train_engines), ('val', val_engines), ('test', test_engines)]:
    with open(f'data/slm/{split}.jsonl') as f:
        for line in f:
            ex = json.loads(line)
            engines.add(ex['meta']['engine'])

print(f'Train engines: {len(train_engines)}')
print(f'Val engines: {len(val_engines)}')
print(f'Test engines: {len(test_engines)}')
print(f'Train ∩ Val: {len(train_engines & val_engines)}')
print(f'Train ∩ Test: {len(train_engines & test_engines)}')
print(f'Val ∩ Test: {len(val_engines & test_engines)}')
print('\n✓ No engine leakage between splits' if not (train_engines & val_engines or train_engines & test_engines or val_engines & test_engines) else '✗ LEAKAGE DETECTED')

Train engines: 80
Val engines: 20
Test engines: 100
Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0

✓ No engine leakage between splits
